# LeetCode #128 — Longest Consecutive Sequence: hash set và "chỉ đếm từ đầu dãy"

**Mức độ:** Medium · **Chủ đề:** Array, Hash Table, Union Find

Skeleton trên LeetCode:

```python
class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Cách dễ: **sắp xếp** rồi đếm — `O(n log n)`, chưa đạt yêu cầu `O(n)`.
2. Ý tưởng chính: dùng **set**, chỉ bắt đầu đếm tại số `x` mà `x − 1` **không** có trong set (đầu một dãy).
3. Chứng minh vì sao tổng thời gian là `O(n)` dù có vòng `while` lồng bên trong.
4. Cách khác: **Union-Find** và hash map "độ dài đoạn tại hai đầu".

# 1. Bài toán là gì?

Cho mảng số nguyên `nums` **chưa sắp xếp**, trả về độ dài của **dãy số nguyên liên tiếp** dài nhất (các phần tử không cần đứng liền nhau trong mảng). Phải chạy trong **`O(n)`**.

**Constraints (theo đề):**

- `0 <= nums.length <= 10^5`
- `-10^9 <= nums[i] <= 10^9`

---

## Ví dụ

```text
[100, 4, 200, 1, 3, 2]              ->  4      (1, 2, 3, 4)
[0, 3, 7, 2, 5, 8, 4, 6, 0, 1]      ->  9      (0 .. 8)
[1, 0, 1, 2]                        ->  3      (0, 1, 2) — phần tử trùng không tính hai lần
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bảng băm (hash table) — `dict` và `set`
2. Sắp xếp (sorting)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [1]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [2]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 252.88 ms | trên set: 0.077 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [3]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [4]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.2 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [5]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [6]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [7]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.3 Áp dụng vào bài này

**Sắp xếp** cho `O(n log n)` (đếm các đoạn liên tiếp). Để đạt `O(n)`: đưa mọi số vào **set**, và chỉ bắt đầu đếm từ những số `x` mà `x − 1` **không** có trong set (đầu dãy) — mỗi lần kiểm tra `in set` là `O(1)`.

# 3. Baseline: sắp xếp rồi đếm

Sau khi sắp xếp, các số liên tiếp đứng cạnh nhau. Bỏ qua phần tử trùng; đặt lại bộ đếm khi có "khoảng trống".

In [8]:
from typing import List
import random, time

def longest_sort(nums: List[int]) -> int:
    if not nums:
        return 0
    nums = sorted(set(nums))
    best = cur = 1
    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1] + 1:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best

print(longest_sort([100, 4, 200, 1, 3, 2]), longest_sort([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]), longest_sort([1, 0, 1, 2]))

4 9 3


`O(n log n)`. Thực tế đủ nhanh, nhưng đề yêu cầu `O(n)`.

# 4. Cách ngây thơ với set: đếm từ **mọi** số

Với mỗi `x`, đếm `x, x+1, x+2, …` có trong set. Đúng, nhưng xấu nhất `O(n²)`: với `1, 2, …, n`, bắt đầu từ `1` đếm `n` bước, từ `2` đếm `n − 1` bước…

In [9]:
def longest_naive_set(nums: List[int]):
    s = set(nums)
    best, steps = 0, 0
    for x in s:
        y = x
        while y in s:
            y += 1
            steps += 1
        best = max(best, y - x)
    return best, steps

print(longest_naive_set(list(range(1, 2001))))

(2000, 2001000)


# 5. Ý tưởng chính: chỉ đếm từ **đầu dãy**

`x` là **đầu** của một dãy liên tiếp ⇔ `x − 1` **không** có trong set. Chỉ từ những `x` đó mới đếm lên.

```text
s = set(nums)
for x in s:
    if x - 1 not in s:            # x là đầu dãy
        y = x
        while y + 1 in s: y += 1
        best = max(best, y - x + 1)
```

In [10]:
def longest_set(nums: List[int]):
    s = set(nums)
    best, steps = 0, 0
    for x in s:
        if x - 1 not in s:
            y = x
            while y + 1 in s:
                y += 1
                steps += 1
            best = max(best, y - x + 1)
    return best, steps

print(longest_set(list(range(1, 2001))))
print(longest_set([100, 4, 200, 1, 3, 2]))

(2000, 1999)
(4, 3)


Cùng input `1..2000`: cách ngây thơ tốn hơn 2 triệu bước, cách "đầu dãy" chỉ 1999 bước.

# 6. Mô phỏng

In [11]:
def trace_set(nums):
    s = set(nums)
    print("set =", sorted(s))
    best = 0
    for x in sorted(s):                         # sắp xếp chỉ để in dễ đọc
        if x - 1 in s:
            print(f"  {x}: có {x - 1} trong set -> không phải đầu dãy, bỏ qua")
            continue
        y = x
        while y + 1 in s:
            y += 1
        print(f"  {x}: đầu dãy -> đếm tới {y}, độ dài {y - x + 1}")
        best = max(best, y - x + 1)
    print("Kết quả:", best)
    return best

_ = trace_set([100, 4, 200, 1, 3, 2])

set = [1, 2, 3, 4, 100, 200]
  1: đầu dãy -> đếm tới 4, độ dài 4
  2: có 1 trong set -> không phải đầu dãy, bỏ qua
  3: có 2 trong set -> không phải đầu dãy, bỏ qua
  4: có 3 trong set -> không phải đầu dãy, bỏ qua
  100: đầu dãy -> đếm tới 100, độ dài 1
  200: đầu dãy -> đếm tới 200, độ dài 1
Kết quả: 4


# 7. Cách 3: hash map lưu độ dài đoạn tại hai đầu

Duyệt từng số mới `x`; nếu `x−1` là **đầu phải** của một đoạn độ dài `L` và `x+1` là **đầu trái** của một đoạn độ dài `R`, thì `x` nối chúng thành đoạn độ dài `L + R + 1`. Chỉ cần cập nhật độ dài ở **hai đầu mút** mới.

In [12]:
def longest_boundary_map(nums: List[int]) -> int:
    length = {}                     # length[v] chỉ đúng nếu v là đầu mút của đoạn chứa nó
    best = 0
    for x in nums:
        if x in length:
            continue
        L = length.get(x - 1, 0)
        R = length.get(x + 1, 0)
        total = L + R + 1
        length[x] = total
        length[x - L] = total       # đầu trái mới
        length[x + R] = total       # đầu phải mới
        best = max(best, total)
    return best

print(longest_boundary_map([100, 4, 200, 1, 3, 2]), longest_boundary_map([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))

4 9


# 8. Cách 4: Union-Find

Mỗi số là một đỉnh; hợp `x` với `x + 1` nếu cả hai có mặt. Đáp án là kích thước **thành phần liên thông** lớn nhất. Gần `O(n)` với nén đường + hợp theo kích thước.

In [13]:
def longest_union_find(nums: List[int]) -> int:
    parent, size = {}, {}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]     # nén đường (halving)
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
    for x in nums:
        if x not in parent:
            parent[x], size[x] = x, 1
    for x in parent:
        if x + 1 in parent:
            union(x, x + 1)
    return max(size[find(x)] for x in parent) if parent else 0

print(longest_union_find([100, 4, 200, 1, 3, 2]), longest_union_find([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))

4 9


# 9. Phiên bản đúng format LeetCode

In [14]:
class Solution:
    def longestConsecutive(self, nums: List[int]) -> int:
        s = set(nums)
        best = 0
        for x in s:
            if x - 1 not in s:
                y = x
                while y + 1 in s:
                    y += 1
                best = max(best, y - x + 1)
        return best

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.longestConsecutive([100, 4, 200, 1, 3, 2]), 4)
check("Ví dụ 2", sol.longestConsecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]), 9)
check("Ví dụ 3", sol.longestConsecutive([1, 0, 1, 2]), 3)
check("Rỗng", sol.longestConsecutive([]), 0)
check("Số âm", sol.longestConsecutive([-1, -2, 0, 5]), 3)
check("Cận 10^9", sol.longestConsecutive([10**9, -10**9, 10**9 - 1]), 2)

random.seed(128)
for _ in range(5000):
    nums = [random.randint(-15, 15) for _ in range(random.randint(0, 20))]
    expected = longest_sort(nums)
    assert sol.longestConsecutive(nums) == expected == longest_boundary_map(nums) == longest_union_find(nums)
print("5000 mảng ngẫu nhiên: 4 cách khớp nhau")

big = list(range(10**5)); random.shuffle(big)
t0 = time.time(); r = sol.longestConsecutive(big); t1 = time.time()
print(f"10^5 số hoán vị ngẫu nhiên: kết quả {r}, {t1 - t0:.3f}s")

Ví dụ 1            PASS  4
Ví dụ 2            PASS  9
Ví dụ 3            PASS  3
Rỗng               PASS  0
Số âm              PASS  3
Cận 10^9           PASS  2


5000 mảng ngẫu nhiên: 4 cách khớp nhau


10^5 số hoán vị ngẫu nhiên: kết quả 100000, 0.065s


# 10. Vì sao đúng? Độ phức tạp?

**Đúng:** mọi dãy liên tiếp cực đại `a, a+1, …, b` có đúng một đầu `a` (vì `a − 1 ∉ s`). Từ `a`, vòng `while` đếm đúng tới `b`. Các `x` không phải đầu dãy bị bỏ qua — không làm mất dãy nào.

**`O(n)`:** vòng `while y + 1 in s` chỉ chạy **bên trong** dãy, bắt đầu từ đầu dãy. Mỗi phần tử của set thuộc đúng một dãy cực đại, nên được bước qua bởi `while` **tối đa một lần**. Tổng số bước `while` ≤ `n`; cộng với `n` lần kiểm tra `x − 1 in s` → `O(n)`.

```text
Time:  O(n) trung bình (thao tác set O(1) trung bình)
Space: O(n)
```

# 11. Những lỗi dễ mắc

## Lỗi 1: Không bỏ trùng khi sắp xếp

In [16]:
def wrong_sort_no_dedup(nums):
    if not nums:
        return 0
    nums = sorted(nums)                           # SAI: thiếu set()
    best = cur = 1
    for i in range(1, len(nums)):
        if nums[i] == nums[i - 1] + 1:
            cur += 1
            best = max(best, cur)
        else:
            cur = 1
    return best

print("Sai :", wrong_sort_no_dedup([1, 2, 0, 1]), "| Đúng:", sol.longestConsecutive([1, 2, 0, 1]))

Sai : 2 | Đúng: 3


Sau khi sắp xếp `[0, 1, 1, 2]`, cặp `(1, 1)` không thỏa `+1` nên bộ đếm bị đặt lại — trong khi đúng ra phải **bỏ qua** phần tử trùng.

## Lỗi 2: Duyệt trên `nums` thay vì trên `set`

Với nhiều bản trùng của đầu dãy (vd. `[1]*50000 + list(range(2, 50001))`), duyệt `for x in nums` sẽ đếm lại cả dãy **mỗi lần gặp** số `1` → `O(n²)`.

In [17]:
def slow_iterate_nums(nums):
    s = set(nums)
    best, steps = 0, 0
    for x in nums:                                # chậm: lặp lại các bản trùng
        if x - 1 not in s:
            y = x
            while y + 1 in s:
                y += 1
                steps += 1
            best = max(best, y - x + 1)
    return best, steps

nums = [1] * 2000 + list(range(2, 2001))
print("Duyệt nums:", slow_iterate_nums(nums), "| duyệt set:", longest_set(nums))

Duyệt nums: (2000, 3998000) | duyệt set: (2000, 1999)


## Lỗi 3: Quên trường hợp mảng rỗng

`max()` trên dãy rỗng gây lỗi; `best = 0` khởi tạo sẵn thì an toàn.

## Lỗi 4: Nghĩ rằng "có `while` lồng trong `for` nên là `O(n²)`"

Lập luận đúng phải đếm **tổng** số bước của vòng trong qua cả chương trình (phân tích khấu hao — amortized), không phải nhân số vòng ngoài với số vòng trong xấu nhất.

# 12. Mẹo nhận diện pattern

- **"Dãy liên tiếp / đoạn số" trên dữ liệu chưa sắp xếp, yêu cầu `O(n)`** → hash set + "chỉ bắt đầu tại điểm đầu".
- **Gộp các đoạn khi thêm phần tử** → hash map độ dài tại đầu mút, hoặc Union-Find.

Bài liên quan: **#298 Binary Tree Longest Consecutive Sequence**, **#549 … II**, **#2501 Longest Square Streak in an Array**, **#41 First Missing Positive** (Hard), **#352 Data Stream as Disjoint Intervals** (Hard).

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `[9, 1, 4, 7, 3, -1, 0, 5, 8, -1, 6]`, những số nào là "đầu dãy"? Kết quả?

## Bài 2 — Điền chỗ trống

In [18]:
def exercise_2_longest_consecutive(nums: List[int]) -> int:
    s = set(nums)
    best = 0
    for x in s:
        # TODO: chỉ đếm khi x là đầu dãy
        pass
    return best


# assert exercise_2_longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
# assert exercise_2_longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
# assert exercise_2_longest_consecutive([]) == 0

## Bài 3 — Biến thể: trả về chính dãy dài nhất

Nếu có nhiều dãy cùng độ dài, trả về dãy có **số đầu nhỏ nhất**.

```python
longest_run([100, 4, 200, 1, 3, 2]) -> [1, 2, 3, 4]
longest_run([5, 6, 1, 2])           -> [1, 2]
```

In [19]:
def exercise_3_longest_run(nums: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_3_longest_run([100, 4, 200, 1, 3, 2]) == [1, 2, 3, 4]
# assert exercise_3_longest_run([5, 6, 1, 2]) == [1, 2]
# assert exercise_3_longest_run([]) == []

## Bài 4 — Biến thể: Longest Square Streak (LeetCode #2501)

Tìm độ dài dãy con dài nhất (sau khi sắp xếp tăng) sao cho mỗi phần tử là **bình phương** của phần tử trước. Độ dài phải ≥ 2, ngược lại trả `-1`.

```python
longest_square_streak([4, 3, 6, 16, 8, 2]) -> 3     # 2, 4, 16
longest_square_streak([2, 3, 5, 6, 7])     -> -1
```

Gợi ý nhanh: giống bài chính, "đầu dãy" là `x` mà `√x` không phải số nguyên có trong set.

In [20]:
def exercise_4_longest_square_streak(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_longest_square_streak([4, 3, 6, 16, 8, 2]) == 3
# assert exercise_4_longest_square_streak([2, 3, 5, 6, 7]) == -1

## Bài 5 — Mini challenge: phiên bản luồng (stream)

Thiết kế lớp `Streak` với `add(x)` và `longest()` — mỗi thao tác `O(1)` trung bình. Dùng ý tưởng "độ dài tại đầu mút" (mục 7).

In [21]:
class exercise_5_Streak:
    def __init__(self):
        # TODO
        pass

    def add(self, x: int) -> None:
        # TODO
        pass

    def longest(self) -> int:
        # TODO
        pass


# st = exercise_5_Streak(); [st.add(v) for v in [5, 1, 3]]; assert st.longest() == 1
# st = exercise_5_Streak(); [st.add(v) for v in [5, 1, 3, 2, 4]]; assert st.longest() == 5
# st = exercise_5_Streak(); [st.add(v) for v in [7, 7, 8]]; assert st.longest() == 2

# 14. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_set` sau khi tự làm.

**Bài 3**
- Mức 1: như bài chính, lưu `(độ dài, đầu)` tốt nhất; so sánh `(-len, start)`.

**Bài 4**
- Mức 1: `r = isqrt(x)`; `x` là đầu dãy nếu `r * r != x` hoặc `r not in s`.
- Mức 2: từ đầu dãy, nhân bình phương liên tục khi còn trong set.

**Bài 5**
- Mức 1: chính là `longest_boundary_map`, tách thành `add` cập nhật `length` và `best`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [22]:
_ = trace_set([9, 1, 4, 7, 3, -1, 0, 5, 8, -1, 6])

set = [-1, 0, 1, 3, 4, 5, 6, 7, 8, 9]
  -1: đầu dãy -> đếm tới 1, độ dài 3
  0: có -1 trong set -> không phải đầu dãy, bỏ qua
  1: có 0 trong set -> không phải đầu dãy, bỏ qua
  3: đầu dãy -> đếm tới 9, độ dài 7
  4: có 3 trong set -> không phải đầu dãy, bỏ qua
  5: có 4 trong set -> không phải đầu dãy, bỏ qua
  6: có 5 trong set -> không phải đầu dãy, bỏ qua
  7: có 6 trong set -> không phải đầu dãy, bỏ qua
  8: có 7 trong set -> không phải đầu dãy, bỏ qua
  9: có 8 trong set -> không phải đầu dãy, bỏ qua
Kết quả: 7


## Đáp án Bài 2

In [23]:
def answer_2_longest_consecutive(nums: List[int]) -> int:
    return longest_set(nums)[0]

assert answer_2_longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert answer_2_longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]) == 9
assert answer_2_longest_consecutive([]) == 0
print("Bài 2 PASS (giống longest_set ở mục 5)")

Bài 2 PASS (giống longest_set ở mục 5)


## Đáp án Bài 3

In [24]:
def answer_3_longest_run(nums: List[int]) -> List[int]:
    s = set(nums)
    best = (0, 0)                               # (-độ dài, đầu)
    for x in s:
        if x - 1 not in s:
            y = x
            while y + 1 in s:
                y += 1
            best = min(best, (-(y - x + 1), x))
    length, start = -best[0], best[1]
    return list(range(start, start + length))

assert answer_3_longest_run([100, 4, 200, 1, 3, 2]) == [1, 2, 3, 4]
assert answer_3_longest_run([5, 6, 1, 2]) == [1, 2]
assert answer_3_longest_run([]) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [25]:
from math import isqrt

def answer_4_longest_square_streak(nums: List[int]) -> int:
    s = set(nums)
    best = 0
    for x in s:
        r = isqrt(x)
        if r * r == x and r in s:
            continue                            # không phải đầu dãy
        length, y = 1, x
        while y * y in s:
            y = y * y
            length += 1
        best = max(best, length)
    return best if best >= 2 else -1

assert answer_4_longest_square_streak([4, 3, 6, 16, 8, 2]) == 3
assert answer_4_longest_square_streak([2, 3, 5, 6, 7]) == -1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [26]:
class answer_5_Streak:
    def __init__(self):
        self.length = {}
        self.best = 0

    def add(self, x: int) -> None:
        if x in self.length:
            return
        L = self.length.get(x - 1, 0)
        R = self.length.get(x + 1, 0)
        total = L + R + 1
        self.length[x] = self.length[x - L] = self.length[x + R] = total
        self.best = max(self.best, total)

    def longest(self) -> int:
        return self.best

st = answer_5_Streak(); [st.add(v) for v in [5, 1, 3]]; assert st.longest() == 1
st = answer_5_Streak(); [st.add(v) for v in [5, 1, 3, 2, 4]]; assert st.longest() == 5
st = answer_5_Streak(); [st.add(v) for v in [7, 7, 8]]; assert st.longest() == 2
for _ in range(1000):
    st, seen = answer_5_Streak(), []
    for _ in range(random.randint(1, 20)):
        v = random.randint(-10, 10)
        st.add(v); seen.append(v)
        assert st.longest() == longest_sort(seen)
print("Bài 5 PASS (đúng sau mỗi lần add)")

Bài 5 PASS (đúng sau mỗi lần add)


# 16. Tổng kết

> Đưa mọi số vào set. Chỉ bắt đầu đếm tại `x` khi `x − 1 ∉ set` (đầu dãy), rồi đếm lên. Mỗi phần tử được vòng trong đi qua tối đa một lần → `O(n)`.

---

## Checklist trước khi nộp

- [ ] Duyệt trên **set** (không phải `nums`, tránh lặp bản trùng)?
- [ ] Chỉ đếm từ đầu dãy (`x − 1 not in s`)?
- [ ] Mảng rỗng → 0?
- [ ] Giải thích được vì sao tổng là `O(n)`?